# Evoluzione del medagliere per genere — analisi definitiva

Questo notebook riprende il tema di `evoluzione_medagliere_genere_indicatori.ipynb`, ma sostituisce il dataset socio-economico ormai obsoleto (`dataset_olimpiadi_indicatori_ita.csv`) con **`src/santina/dataset_indicatori_definitivo.csv`** e propone un'analisi diversa, centrata sulla **partecipazione femminile** oltre che sulle medaglie.

Dataset usati:

- `data/csv_olimpiadi/Olympic_Athlete_Event_Details.csv` — atleta, evento, sport, medaglia e NOC;
- `data/csv_olimpiadi/Olympic_Athlete_Biography.csv` — sesso degli atleti e nome paese leggibile;
- `src/santina/dataset_indicatori_definitivo.csv` — anno, edizione, nazione e indicatori socio-economici World Bank in italiano.

Le domande guida sono tre:

1. **Come è cresciuto nel tempo il peso delle medaglie femminili?**
2. **Quali paesi costruiscono una parte rilevante del proprio medagliere sulle competizioni femminili?**
3. **I risultati maschili e femminili rispondono agli stessi fattori socio-economici?**

Tutti i grafici sono realizzati con **Altair**. In chiusura vengono lasciate esplicitamente alcune **domande aperte**: gli indicatori disponibili non possono spiegare tutto, e il notebook lo dichiara.

In [1]:
from pathlib import Path
import re

import altair as alt
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
alt.data_transformers.disable_max_rows()

# Palette coerente con gli altri notebook del progetto.
COLORE_M = "#3569a8"
COLORE_F = "#c7437a"
COLORE_MISTO = "#8a8a8a"
SCALA_GENERE = alt.Scale(domain=["Maschile", "Femminile"], range=[COLORE_M, COLORE_F])
SCALA_GENERE_MISTO = alt.Scale(
    domain=["Maschile", "Femminile", "Misto/Altro"],
    range=[COLORE_M, COLORE_F, COLORE_MISTO],
)

## 1. Caricamento dei dati

Il notebook cerca automaticamente la root del progetto, quindi può essere eseguito dalla root o da una sottocartella del repository.

Una nota importante sul dataset degli indicatori: a differenza del vecchio `dataset_olimpiadi_indicatori_ita.csv`, il file definitivo **mantiene i codici NOC storici** (ad esempio `URS` per l'Unione Sovietica, `GDR` per la Germania Est). Questo permette un merge diretto con i dati olimpici, senza rimappature manuali: tutti i paesi che hanno vinto almeno una medaglia estiva tra il 1964 e il 2020 sono presenti nel dataset degli indicatori.

In [2]:
def find_project_root() -> Path:
    expected = Path("data") / "csv_olimpiadi" / "Olympic_Athlete_Event_Details.csv"
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / expected).exists():
            return base
    raise FileNotFoundError("Non trovo la root del progetto con la cartella data/.")

PROJECT_ROOT = find_project_root()
EVENTS_PATH = PROJECT_ROOT / "data" / "csv_olimpiadi" / "Olympic_Athlete_Event_Details.csv"
BIO_PATH = PROJECT_ROOT / "data" / "csv_olimpiadi" / "Olympic_Athlete_Biography.csv"
SOCIO_PATH = PROJECT_ROOT / "src" / "santina" / "dataset_indicatori_definitivo.csv"

events = pd.read_csv(
    EVENTS_PATH,
    usecols=["edition", "country_noc", "sport", "event", "athlete_id", "medal", "isTeamSport"],
)
bio = pd.read_csv(BIO_PATH, usecols=["athlete_id", "sex", "country", "country_noc"])
socio = pd.read_csv(SOCIO_PATH, low_memory=False)

events["athlete_id"] = events["athlete_id"].astype("string")
bio["athlete_id"] = bio["athlete_id"].astype("string")

print("Project root:", PROJECT_ROOT)
print(f"Righe atleta-evento: {len(events):,}")
print(f"Righe biografie: {len(bio):,}")
print(f"Righe indicatori socio-economici: {len(socio):,}")

Project root: C:\Users\smacchiavelli\development\code\master\progettone
Righe atleta-evento: 316,834
Righe biografie: 155,861
Righe indicatori socio-economici: 3,368


## 2. Preparazione: sesso degli atleti, genere degli eventi, deduplica delle squadre

Il file atleta-evento contiene una riga per atleta: negli sport di squadra la stessa medaglia compare una volta per ogni componente. Per ricostruire un medagliere coerente:

- negli sport individuali mantengo le righe-medaglia originali;
- negli sport di squadra conto una sola medaglia per `anno + paese + sport + evento + tipo medaglia`;
- classifico ogni evento come `Maschile`, `Femminile` o `Misto/Altro` leggendo il nome dell'evento;
- il sesso dell'atleta (per l'analisi della partecipazione) viene invece dalle biografie.

Gli eventi misti restano separati: non vengono attribuiti arbitrariamente né al medagliere maschile né a quello femminile.

Come controllo di qualità, la tabella incrociata sotto mostra che gli eventi classificati "Femminile" sono disputati quasi esclusivamente da atlete, e viceversa. In fondo alla sezione confronto anche il totale ricostruito con il medagliere ufficiale (`Totale_Medaglie`) del dataset indicatori.

In [3]:
events["year"] = events["edition"].str.extract(r"(\d{4})", expand=False).astype(int)
is_summer = events["edition"].str.contains("Summer Olympics", case=False, na=False)
events = events.loc[is_summer & events["year"].between(1964, 2020)].copy()

if events["isTeamSport"].dtype == bool:
    events["is_team_sport"] = events["isTeamSport"]
else:
    events["is_team_sport"] = (
        events["isTeamSport"].astype("string").str.strip().str.lower().eq("true")
    )

events = events.merge(bio[["athlete_id", "sex"]], on="athlete_id", how="left")
events["sesso_atleta"] = (
    events["sex"].map({"Male": "Maschile", "Female": "Femminile"}).fillna("Sconosciuto")
)
events["has_medal"] = events["medal"].notna() & events["medal"].astype("string").str.strip().ne("")

def classifica_genere_evento(event: str) -> str:
    text = str(event).lower()
    if re.search(r"\bwomen\b|women's|, women", text):
        return "Femminile"
    if re.search(r"\bmen\b|men's|, men", text):
        return "Maschile"
    return "Misto/Altro"

events["genere_evento"] = events["event"].apply(classifica_genere_evento)

print(f"Righe dopo il filtro Summer 1964-2020: {len(events):,}")
display(pd.crosstab(events["genere_evento"], events["sesso_atleta"]))

Righe dopo il filtro Summer 1964-2020: 179,141


sesso_atleta,Femminile,Maschile
genere_evento,,
Femminile,60312,3
Maschile,1,108599
Misto/Altro,1965,8261


In [4]:
medal_rows = events.loc[events["has_medal"]].copy()

team_medals = (
    medal_rows.loc[medal_rows["is_team_sport"]]
    .drop_duplicates(["year", "country_noc", "sport", "event", "medal"])
)
individual_medals = medal_rows.loc[~medal_rows["is_team_sport"]]
medal_units = pd.concat([individual_medals, team_medals], ignore_index=True)

medals_gender = (
    medal_units.groupby(["year", "country_noc", "genere_evento"], as_index=False)
    .size()
    .pivot_table(
        index=["year", "country_noc"],
        columns="genere_evento",
        values="size",
        fill_value=0,
        aggfunc="sum",
    )
    .reset_index()
    .rename(columns={
        "Femminile": "medaglie_f",
        "Maschile": "medaglie_m",
        "Misto/Altro": "medaglie_misto",
    })
)
for col in ["medaglie_f", "medaglie_m", "medaglie_misto"]:
    if col not in medals_gender.columns:
        medals_gender[col] = 0
medals_gender["medaglie_tot"] = (
    medals_gender[["medaglie_f", "medaglie_m", "medaglie_misto"]].sum(axis=1)
)

# Controllo di coerenza con il medagliere ufficiale del dataset indicatori.
check = medals_gender.merge(
    socio[["Anno", "Codice_NOC", "Totale_Medaglie"]],
    left_on=["year", "country_noc"],
    right_on=["Anno", "Codice_NOC"],
    how="inner",
)
scarto = (check["medaglie_tot"] - check["Totale_Medaglie"]).abs()
print(f"Medaglie-evento dopo la deduplica delle squadre: {len(medal_units):,}")
print(
    f"Coerenza con il medagliere ufficiale: r={check['medaglie_tot'].corr(check['Totale_Medaglie']):.4f}, "
    f"scarto medio assoluto={scarto.mean():.2f} medaglie per paese-edizione"
)

Medaglie-evento dopo la deduplica delle squadre: 11,781
Coerenza con il medagliere ufficiale: r=0.9870, scarto medio assoluto=0.92 medaglie per paese-edizione


## 3. Domanda 1 — Come è cresciuto nel tempo il peso delle medaglie femminili?

Il peso delle medaglie femminili va letto insieme alla **partecipazione**: la quota di medaglie disponibili per le donne dipende dal numero di eventi femminili in programma, che a sua volta cresce insieme al numero di atlete. Alcune tappe utili per la lettura: la maratona femminile debutta nel 1984, e Londra 2012 è la prima edizione in cui le donne gareggiano in tutti gli sport del programma.

Il primo grafico mostra i partecipanti unici per edizione, separati per sesso.

In [5]:
partecipanti = (
    events.loc[events["sesso_atleta"].isin(["Maschile", "Femminile"])]
    .drop_duplicates(subset=["year", "athlete_id"])
    .groupby(["year", "sesso_atleta"], as_index=False)
    .size()
    .rename(columns={"size": "atleti"})
)

partecipazione_chart = (
    alt.Chart(partecipanti)
    .mark_line(point=True, strokeWidth=2)
    .encode(
        x=alt.X("year:O", title="Anno"),
        y=alt.Y("atleti:Q", title="Atleti partecipanti (unici per edizione)"),
        color=alt.Color("sesso_atleta:N", title="Sesso", scale=SCALA_GENERE),
        tooltip=[
            alt.Tooltip("year:O", title="Anno"),
            alt.Tooltip("sesso_atleta:N", title="Sesso"),
            alt.Tooltip("atleti:Q", title="Atleti", format=",.0f"),
        ],
    )
    .properties(width=760, height=380, title="Partecipanti alle Olimpiadi estive per sesso, 1964-2020")
)

partecipazione_chart

alt.Chart(...)

### Quota di partecipazione e quota di medaglie viaggiano insieme?

Qui confronto due quote calcolate sullo stesso denominatore concettuale (femminile su femminile + maschile, eventi misti esclusi):

- la **quota di atlete** tra i partecipanti;
- la **quota di medaglie femminili** sul totale delle medaglie di genere.

Se le due curve coincidono, la crescita del medagliere femminile è essenzialmente la crescita delle opportunità di gara; se divergono, c'è qualcosa in più da spiegare.

In [6]:
quota_part = (
    partecipanti.pivot_table(index="year", columns="sesso_atleta", values="atleti")
    .reset_index()
)
quota_part["quota"] = quota_part["Femminile"] / (quota_part["Femminile"] + quota_part["Maschile"])
quota_part["misura"] = "Quota atlete tra i partecipanti"

medaglie_anno = (
    medal_units.groupby(["year", "genere_evento"], as_index=False)
    .size()
    .pivot_table(index="year", columns="genere_evento", values="size", fill_value=0)
    .reset_index()
)
medaglie_anno["quota"] = medaglie_anno["Femminile"] / (
    medaglie_anno["Femminile"] + medaglie_anno["Maschile"]
)
medaglie_anno["misura"] = "Quota medaglie femminili"

quote = pd.concat(
    [quota_part[["year", "quota", "misura"]], medaglie_anno[["year", "quota", "misura"]]],
    ignore_index=True,
)

linee_quote = (
    alt.Chart(quote)
    .mark_line(point=True, strokeWidth=2.5, color=COLORE_F)
    .encode(
        x=alt.X("year:O", title="Anno"),
        y=alt.Y(
            "quota:Q",
            title="Quota femminile",
            axis=alt.Axis(format="%"),
            scale=alt.Scale(domain=[0, 0.6]),
        ),
        strokeDash=alt.StrokeDash("misura:N", title=""),
        tooltip=[
            alt.Tooltip("year:O", title="Anno"),
            alt.Tooltip("misura:N", title="Misura"),
            alt.Tooltip("quota:Q", title="Quota", format=".1%"),
        ],
    )
)
parita = (
    alt.Chart(pd.DataFrame({"quota": [0.5]}))
    .mark_rule(strokeDash=[4, 4], color="#999999")
    .encode(y="quota:Q")
)

quote_confronto_chart = (linee_quote + parita).properties(
    width=760, height=380,
    title="Quota di atlete e quota di medaglie femminili, 1964-2020",
)

quote_confronto_chart

alt.LayerChart(...)

### Composizione del medagliere per genere dell'evento

L'area normalizzata mostra come si è ricomposto il medagliere complessivo tra eventi maschili, femminili e misti.

In [7]:
medaglie_long = (
    medal_units.groupby(["year", "genere_evento"], as_index=False)
    .size()
    .rename(columns={"size": "medaglie"})
)

area_genere_chart = (
    alt.Chart(medaglie_long)
    .mark_area()
    .encode(
        x=alt.X("year:O", title="Anno"),
        y=alt.Y("medaglie:Q", title="Quota del medagliere", stack="normalize", axis=alt.Axis(format="%")),
        color=alt.Color("genere_evento:N", title="Genere evento", scale=SCALA_GENERE_MISTO),
        tooltip=[
            alt.Tooltip("year:O", title="Anno"),
            alt.Tooltip("genere_evento:N", title="Genere evento"),
            alt.Tooltip("medaglie:Q", title="Medaglie", format=",.0f"),
        ],
    )
    .properties(width=760, height=360, title="Composizione del medagliere olimpico estivo per genere dell'evento")
)

area_genere_chart

alt.Chart(...)

### Resa: medaglie ogni 100 partecipanti

Un ultimo angolo sulla stessa domanda: quante medaglie (di eventi del proprio genere) vengono assegnate ogni 100 partecipanti di quel sesso? Se la resa femminile è simile a quella maschile, la crescita del medagliere femminile è spiegata dalla crescita della partecipazione, non da una diversa "densità" di medaglie.

*Attenzione*: le medaglie di squadra sono contate una volta per squadra, quindi la resa non è la probabilità individuale di vincere una medaglia; il confronto tra i due generi resta però coerente.

In [8]:
resa = medaglie_anno[["year", "Femminile", "Maschile"]].merge(
    quota_part[["year", "Femminile", "Maschile"]],
    on="year",
    suffixes=("_medaglie", "_atleti"),
)
resa["Femminile"] = 100 * resa["Femminile_medaglie"] / resa["Femminile_atleti"]
resa["Maschile"] = 100 * resa["Maschile_medaglie"] / resa["Maschile_atleti"]

resa_long = resa.melt(
    id_vars="year", value_vars=["Maschile", "Femminile"],
    var_name="genere", value_name="medaglie_per_100",
)

resa_chart = (
    alt.Chart(resa_long)
    .mark_line(point=True, strokeWidth=2)
    .encode(
        x=alt.X("year:O", title="Anno"),
        y=alt.Y("medaglie_per_100:Q", title="Medaglie ogni 100 partecipanti"),
        color=alt.Color("genere:N", title="Genere", scale=SCALA_GENERE),
        tooltip=[
            alt.Tooltip("year:O", title="Anno"),
            alt.Tooltip("genere:N", title="Genere"),
            alt.Tooltip("medaglie_per_100:Q", title="Medaglie / 100 atleti", format=".1f"),
        ],
    )
    .properties(width=760, height=360, title="Resa in medaglie della partecipazione, per genere")
)

resa_chart

alt.Chart(...)

## 4. Domanda 2 — Quali paesi costruiscono il medagliere sulle competizioni femminili?

Qui aggrego l'intero periodo 1964-2020 e calcolo, per ogni paese, la **quota femminile** del medagliere (femminile su femminile + maschile, misti esclusi). Considero solo i paesi con almeno 25 medaglie di genere, per evitare quote instabili calcolate su numeri piccoli.

La linea tratteggiata al 50% segna la parità: i paesi a destra hanno vinto più con le donne che con gli uomini.

In [9]:
noc_nome = socio.sort_values("Anno").groupby("Codice_NOC")["Nazione"].last()

paesi = medals_gender.groupby("country_noc", as_index=False)[
    ["medaglie_f", "medaglie_m", "medaglie_misto"]
].sum()
paesi["medaglie_fm"] = paesi["medaglie_f"] + paesi["medaglie_m"]
paesi["quota_f"] = paesi["medaglie_f"] / paesi["medaglie_fm"]
paesi["paese"] = paesi["country_noc"].map(noc_nome).fillna(paesi["country_noc"])

paesi_rilevanti = (
    paesi.loc[paesi["medaglie_fm"] >= 25]
    .sort_values("quota_f", ascending=False)
    .reset_index(drop=True)
)
print(f"Paesi con almeno 25 medaglie di genere (1964-2020): {len(paesi_rilevanti)}")

punti_paesi = (
    alt.Chart(paesi_rilevanti)
    .mark_circle(opacity=0.85, stroke="#333333", strokeWidth=0.4, color=COLORE_F)
    .encode(
        y=alt.Y("paese:N", sort=paesi_rilevanti["paese"].tolist(), title=None),
        x=alt.X("quota_f:Q", title="Quota femminile del medagliere 1964-2020", axis=alt.Axis(format="%")),
        size=alt.Size("medaglie_fm:Q", title="Medaglie M+F", scale=alt.Scale(range=[30, 700])),
        tooltip=[
            alt.Tooltip("paese:N", title="Paese"),
            alt.Tooltip("medaglie_f:Q", title="Medaglie femminili", format=",.0f"),
            alt.Tooltip("medaglie_m:Q", title="Medaglie maschili", format=",.0f"),
            alt.Tooltip("quota_f:Q", title="Quota femminile", format=".1%"),
        ],
    )
)
parita_v = (
    alt.Chart(pd.DataFrame({"quota_f": [0.5]}))
    .mark_rule(strokeDash=[4, 4], color="#999999")
    .encode(x="quota_f:Q")
)

paesi_quota_chart = (punti_paesi + parita_v).properties(
    width=680,
    height=16 * len(paesi_rilevanti),
    title="Quanto pesa il medagliere femminile nei paesi con almeno 25 medaglie",
)

paesi_quota_chart

Paesi con almeno 25 medaglie di genere (1964-2020): 62


alt.LayerChart(...)

### La quota femminile per paese: prima e dopo il 2000

La quota aggregata nasconde traiettorie diverse. Confronto la quota femminile di ogni paese nel periodo **1964-1996** con quella del periodo **2000-2020** (almeno 10 medaglie di genere in ciascun periodo). I paesi sopra la diagonale hanno spostato il proprio medagliere verso le donne.

In [10]:
mg = medals_gender.copy()
mg["periodo"] = np.where(mg["year"] <= 1996, "1964-1996", "2000-2020")
per_periodo = mg.groupby(["country_noc", "periodo"], as_index=False)[
    ["medaglie_f", "medaglie_m"]
].sum()
per_periodo["fm"] = per_periodo["medaglie_f"] + per_periodo["medaglie_m"]
per_periodo["quota_f"] = per_periodo["medaglie_f"] / per_periodo["fm"]

early = per_periodo.loc[per_periodo["periodo"] == "1964-1996", ["country_noc", "quota_f", "fm"]]
late = per_periodo.loc[per_periodo["periodo"] == "2000-2020", ["country_noc", "quota_f", "fm"]]
confronto = early.merge(late, on="country_noc", suffixes=("_early", "_late"))
confronto = confronto.loc[(confronto["fm_early"] >= 10) & (confronto["fm_late"] >= 10)].copy()
confronto["paese"] = confronto["country_noc"].map(noc_nome).fillna(confronto["country_noc"])
confronto["delta"] = confronto["quota_f_late"] - confronto["quota_f_early"]

punti_periodi = (
    alt.Chart(confronto)
    .mark_circle(opacity=0.8, stroke="#333333", strokeWidth=0.4, color=COLORE_F)
    .encode(
        x=alt.X("quota_f_early:Q", title="Quota femminile 1964-1996", axis=alt.Axis(format="%")),
        y=alt.Y("quota_f_late:Q", title="Quota femminile 2000-2020", axis=alt.Axis(format="%")),
        size=alt.Size("fm_late:Q", title="Medaglie M+F 2000-2020", scale=alt.Scale(range=[30, 700])),
        tooltip=[
            alt.Tooltip("paese:N", title="Paese"),
            alt.Tooltip("quota_f_early:Q", title="Quota 1964-1996", format=".1%"),
            alt.Tooltip("quota_f_late:Q", title="Quota 2000-2020", format=".1%"),
            alt.Tooltip("delta:Q", title="Variazione", format="+.1%"),
        ],
    )
)
diagonale = (
    alt.Chart(pd.DataFrame({"x": [0.0, 1.0], "y": [0.0, 1.0]}))
    .mark_line(strokeDash=[5, 5], color="#777777")
    .encode(x="x:Q", y="y:Q")
)

periodi_scatter = (diagonale + punti_periodi).properties(
    width=560, height=520,
    title="Chi ha spostato il medagliere verso le donne dopo il 2000?",
).interactive()

periodi_scatter

alt.LayerChart(...)

## 5. Domanda 3 — Maschile e femminile rispondono agli stessi fattori socio-economici?

Costruisco un panel paese-edizione usando il dataset degli indicatori come base, così restano anche i paesi che in una certa edizione hanno zero medaglie: le correlazioni non devono limitarsi ai soli vincitori.

Scelte metodologiche:

- come target uso `log(1 + medaglie)`, per evitare che pochi paesi dominanti schiaccino tutto;
- gli indicatori più asimmetrici (PIL, PIL pro capite, popolazione) vengono trasformati in logaritmo;
- le correlazioni sono **descrittive**: misurano associazioni, non causalità.

In [11]:
socio_panel = socio.rename(
    columns={"Anno": "year", "Codice_NOC": "country_noc", "Nazione": "paese"}
).copy()
socio_panel["year"] = socio_panel["year"].astype(int)

panel = socio_panel.merge(
    medals_gender[["year", "country_noc", "medaglie_f", "medaglie_m", "medaglie_misto", "medaglie_tot"]],
    on=["year", "country_noc"],
    how="left",
)
for col in ["medaglie_f", "medaglie_m", "medaglie_misto", "medaglie_tot"]:
    panel[col] = panel[col].fillna(0)

panel["medaglie_fm"] = panel["medaglie_f"] + panel["medaglie_m"]
panel["quota_f"] = np.where(panel["medaglie_fm"] > 0, panel["medaglie_f"] / panel["medaglie_fm"], np.nan)

panel["log_PIL_Assoluto"] = np.log1p(panel["PIL_Assoluto_USD"])
panel["log_PIL_Pro_Capite"] = np.log1p(panel["PIL_Pro_Capite_USD"])
panel["log_Popolazione"] = np.log1p(panel["Popolazione_Totale"])
panel["log_medaglie_f"] = np.log1p(panel["medaglie_f"])
panel["log_medaglie_m"] = np.log1p(panel["medaglie_m"])

INDICATORI = {
    "log_PIL_Assoluto": "log(PIL assoluto)",
    "log_PIL_Pro_Capite": "log(PIL pro capite)",
    "log_Popolazione": "log(Popolazione)",
    "Aspettativa_di_Vita": "Aspettativa di vita",
    "Tasso_Mortalita_Infantile": "Mortalita' infantile",
    "Tasso_Urbanizzazione_perc": "Urbanizzazione %",
    "Popolazione_Eta_Lavorativa_perc": "Eta' lavorativa %",
    "Crescita_Demografica_perc": "Crescita demografica %",
    "Crescita_PIL_perc_annua": "Crescita PIL %",
    "Densita_Popolazione": "Densita' popolazione",
    "Iscrizioni_Scuola_Primaria_perc": "Iscrizione primaria %",
    "Spesa_Militare_perc_PIL": "Spesa militare % PIL",
    "Interscambio_Commerciale_perc_PIL": "Apertura commerciale % PIL",
}
for col in INDICATORI:
    panel[col] = pd.to_numeric(panel[col], errors="coerce")

print(f"Osservazioni paese-edizione nel panel: {len(panel):,}")
print(f"Paesi distinti: {panel['country_noc'].nunique():,}")

Osservazioni paese-edizione nel panel: 3,368
Paesi distinti: 218


### Correlazioni per genere, fianco a fianco

Per ogni indicatore calcolo la correlazione di Pearson con `log(1 + medaglie maschili)` e con `log(1 + medaglie femminili)`. Se i due punti di un indicatore sono vicini, quel fattore "spiega" i due medaglieri allo stesso modo.

In [12]:
records = []
for col, label in INDICATORI.items():
    for target, genere in [("log_medaglie_m", "Maschile"), ("log_medaglie_f", "Femminile")]:
        tmp = panel[[col, target]].replace([np.inf, -np.inf], np.nan).dropna()
        if len(tmp) >= 30 and tmp[col].nunique() > 1:
            records.append({
                "indicatore": label,
                "genere": genere,
                "r": tmp[col].corr(tmp[target]),
                "n": len(tmp),
            })
corr_genere = pd.DataFrame(records)

ordine_indicatori = (
    corr_genere.loc[corr_genere["genere"] == "Femminile"]
    .sort_values("r", ascending=False)["indicatore"]
    .tolist()
)

punti_corr = (
    alt.Chart(corr_genere)
    .mark_circle(size=110, opacity=0.9, stroke="#333333", strokeWidth=0.4)
    .encode(
        y=alt.Y("indicatore:N", sort=ordine_indicatori, title=None),
        x=alt.X("r:Q", title="Correlazione con log(1 + medaglie)", scale=alt.Scale(domain=[-0.4, 0.7])),
        color=alt.Color("genere:N", title="Medagliere", scale=SCALA_GENERE),
        tooltip=[
            alt.Tooltip("indicatore:N", title="Indicatore"),
            alt.Tooltip("genere:N", title="Medagliere"),
            alt.Tooltip("r:Q", title="r di Pearson", format=".3f"),
            alt.Tooltip("n:Q", title="Osservazioni", format=",.0f"),
        ],
    )
)
zero_v = (
    alt.Chart(pd.DataFrame({"r": [0.0]}))
    .mark_rule(color="#999999")
    .encode(x="r:Q")
)

corr_dot_chart = (zero_v + punti_corr).properties(
    width=680, height=380,
    title="Gli stessi fattori? Correlazioni con il medagliere maschile e femminile",
)

corr_dot_chart

alt.LayerChart(...)

### Dove le correlazioni divergono

Il grafico seguente isola la **differenza** tra la correlazione femminile e quella maschile: barre verso destra indicano fattori più legati al successo femminile, verso sinistra al successo maschile.

In [13]:
corr_wide = corr_genere.pivot_table(index="indicatore", columns="genere", values="r").reset_index()
corr_wide["differenza"] = corr_wide["Femminile"] - corr_wide["Maschile"]
corr_wide = corr_wide.sort_values("differenza", ascending=False)

corr_diff_chart = (
    alt.Chart(corr_wide)
    .mark_bar()
    .encode(
        y=alt.Y("indicatore:N", sort=corr_wide["indicatore"].tolist(), title=None),
        x=alt.X("differenza:Q", title="r femminile - r maschile"),
        color=alt.condition(alt.datum.differenza > 0, alt.value(COLORE_F), alt.value(COLORE_M)),
        tooltip=[
            alt.Tooltip("indicatore:N", title="Indicatore"),
            alt.Tooltip("Femminile:Q", title="r femminile", format=".3f"),
            alt.Tooltip("Maschile:Q", title="r maschile", format=".3f"),
            alt.Tooltip("differenza:Q", title="Differenza", format="+.3f"),
        ],
    )
    .properties(width=680, height=360, title="Differenza tra correlazione femminile e maschile, per indicatore")
)

corr_diff_chart

alt.Chart(...)

### E la quota femminile del medagliere?

Le correlazioni con il *numero* di medaglie dicono chi vince tanto; la **quota femminile** dice invece come è composto il medagliere. Qui correlo gli indicatori con la quota femminile, considerando solo le osservazioni paese-edizione con almeno 5 medaglie di genere (le quote su numeri piccoli sono rumore).

In [14]:
sub_quota = panel.loc[panel["medaglie_fm"] >= 5]

records_q = []
for col, label in INDICATORI.items():
    tmp = sub_quota[[col, "quota_f"]].replace([np.inf, -np.inf], np.nan).dropna()
    if len(tmp) >= 30 and tmp[col].nunique() > 1:
        records_q.append({"indicatore": label, "r": tmp[col].corr(tmp["quota_f"]), "n": len(tmp)})
corr_quota = pd.DataFrame(records_q).sort_values("r", ascending=False)

quota_corr_chart = (
    alt.Chart(corr_quota)
    .mark_bar(color=COLORE_F)
    .encode(
        y=alt.Y("indicatore:N", sort=corr_quota["indicatore"].tolist(), title=None),
        x=alt.X("r:Q", title="Correlazione con la quota femminile del medagliere"),
        tooltip=[
            alt.Tooltip("indicatore:N", title="Indicatore"),
            alt.Tooltip("r:Q", title="r di Pearson", format=".3f"),
            alt.Tooltip("n:Q", title="Osservazioni", format=",.0f"),
        ],
    )
    .properties(width=680, height=360, title="Quali indicatori si associano a un medagliere piu' femminile?")
)

quota_corr_chart

alt.Chart(...)

### Ricchezza e medaglie, con rette di regressione per genere

Per il periodo più comparabile (2000-2020, quando il programma femminile è ampio) mostro la relazione tra **PIL pro capite** e medaglie, con una retta di regressione separata per genere. Pendenze simili = stessa risposta alla ricchezza.

In [15]:
recente = panel.loc[panel["year"] >= 2000]

scatter_data = recente[
    ["year", "paese", "country_noc", "log_PIL_Pro_Capite", "medaglie_m", "medaglie_f"]
].melt(
    id_vars=["year", "paese", "country_noc", "log_PIL_Pro_Capite"],
    value_vars=["medaglie_m", "medaglie_f"],
    var_name="genere",
    value_name="medaglie",
)
scatter_data["genere"] = scatter_data["genere"].map({"medaglie_m": "Maschile", "medaglie_f": "Femminile"})
scatter_data["log_medaglie"] = np.log1p(scatter_data["medaglie"])
scatter_data = scatter_data.dropna(subset=["log_PIL_Pro_Capite"])

base = alt.Chart(scatter_data)
punti = base.mark_circle(opacity=0.25, size=40).encode(
    x=alt.X("log_PIL_Pro_Capite:Q", title="log(PIL pro capite)", scale=alt.Scale(zero=False)),
    y=alt.Y("log_medaglie:Q", title="log(1 + medaglie)"),
    color=alt.Color("genere:N", title="Medagliere", scale=SCALA_GENERE),
    tooltip=[
        alt.Tooltip("paese:N", title="Paese"),
        alt.Tooltip("year:O", title="Anno"),
        alt.Tooltip("genere:N", title="Genere"),
        alt.Tooltip("medaglie:Q", title="Medaglie", format=",.0f"),
    ],
)
rette = (
    base.transform_regression("log_PIL_Pro_Capite", "log_medaglie", groupby=["genere"])
    .mark_line(strokeWidth=3)
    .encode(
        x="log_PIL_Pro_Capite:Q",
        y="log_medaglie:Q",
        color=alt.Color("genere:N", scale=SCALA_GENERE),
    )
)

pil_scatter_reg = (punti + rette).properties(
    width=760, height=420,
    title="PIL pro capite e medaglie per genere, 2000-2020",
).interactive()

pil_scatter_reg

alt.LayerChart(...)

### La quota femminile cresce con la ricchezza?

Ultimo grafico: la quota femminile del medagliere (2000-2020, almeno 5 medaglie di genere) contro il PIL pro capite. Se lo sviluppo economico favorisse sistematicamente lo sport femminile, i punti dovrebbero salire verso destra.

In [16]:
quota_data = recente.loc[
    (recente["medaglie_fm"] >= 5) & recente["log_PIL_Pro_Capite"].notna(),
    ["year", "paese", "country_noc", "log_PIL_Pro_Capite", "quota_f", "medaglie_fm"],
]

base_q = alt.Chart(quota_data)
punti_q = base_q.mark_circle(opacity=0.55, stroke="#333333", strokeWidth=0.3, color=COLORE_F).encode(
    x=alt.X("log_PIL_Pro_Capite:Q", title="log(PIL pro capite)", scale=alt.Scale(zero=False)),
    y=alt.Y("quota_f:Q", title="Quota femminile del medagliere", axis=alt.Axis(format="%")),
    size=alt.Size("medaglie_fm:Q", title="Medaglie M+F", scale=alt.Scale(range=[20, 500])),
    tooltip=[
        alt.Tooltip("paese:N", title="Paese"),
        alt.Tooltip("year:O", title="Anno"),
        alt.Tooltip("quota_f:Q", title="Quota femminile", format=".1%"),
        alt.Tooltip("medaglie_fm:Q", title="Medaglie M+F", format=",.0f"),
    ],
)
retta_q = (
    base_q.transform_regression("log_PIL_Pro_Capite", "quota_f")
    .mark_line(strokeWidth=3, color="#333333")
    .encode(x="log_PIL_Pro_Capite:Q", y="quota_f:Q")
)

quota_pil_scatter = (punti_q + retta_q).properties(
    width=760, height=420,
    title="Quota femminile del medagliere e PIL pro capite, 2000-2020",
).interactive()

quota_pil_scatter

alt.LayerChart(...)

## 6. Sintesi automatica

Questa cella produce un riepilogo numerico dei punti principali, utile per la relazione.

In [17]:
q_med_64 = float(medaglie_anno.loc[medaglie_anno["year"] == 1964, "quota"].iloc[0])
q_med_20 = float(medaglie_anno.loc[medaglie_anno["year"] == 2020, "quota"].iloc[0])
q_par_64 = float(quota_part.loc[quota_part["year"] == 1964, "quota"].iloc[0])
q_par_20 = float(quota_part.loc[quota_part["year"] == 2020, "quota"].iloc[0])

top3 = paesi_rilevanti.head(3)
top3_txt = ", ".join(
    f"{r.paese} ({r.quota_f:.1%})" for r in top3.itertuples()
)

pil_pc = corr_wide.loc[corr_wide["indicatore"] == "log(PIL pro capite)"].iloc[0]
max_div = corr_wide.loc[corr_wide["differenza"].abs().idxmax()]
top_quota_corr = corr_quota.iloc[0]

print("Sintesi")
print("-------")
print(f"Quota di atlete tra i partecipanti: {q_par_64:.1%} nel 1964 -> {q_par_20:.1%} nel 2020.")
print(f"Quota di medaglie femminili: {q_med_64:.1%} nel 1964 -> {q_med_20:.1%} nel 2020.")
print(f"Paesi con il medagliere piu' femminile (min 25 medaglie): {top3_txt}.")
print(
    f"Correlazione con log(PIL pro capite): r={pil_pc['Femminile']:.3f} per le donne, "
    f"r={pil_pc['Maschile']:.3f} per gli uomini."
)
print(
    f"L'indicatore con la divergenza piu' ampia tra i generi e' {max_div['indicatore']} "
    f"(r femminile - r maschile = {max_div['differenza']:+.3f})."
)
print(
    f"L'indicatore piu' associato a un medagliere femminile e' {top_quota_corr['indicatore']} "
    f"(r={top_quota_corr['r']:.3f})."
)

Sintesi
-------
Quota di atlete tra i partecipanti: 13.0% nel 1964 -> 47.8% nel 2020.
Quota di medaglie femminili: 21.0% nel 1964 -> 48.3% nel 2020.
Paesi con il medagliere piu' femminile (min 25 medaglie): Jamaica (62.5%), Chinese Taipei (61.8%), Netherlands (57.8%).
Correlazione con log(PIL pro capite): r=0.351 per le donne, r=0.327 per gli uomini.
L'indicatore con la divergenza piu' ampia tra i generi e' Apertura commerciale % PIL (r femminile - r maschile = +0.066).
L'indicatore piu' associato a un medagliere femminile e' log(PIL assoluto) (r=0.277).


## 7. Conclusioni e domande aperte

**Cosa emerge dall'analisi.**

- Il peso delle medaglie femminili cresce costantemente dal 1964 al 2020 e la quota di medaglie segue da vicino la quota di partecipazione: gran parte della crescita del medagliere femminile è la crescita delle *opportunità* (più eventi, più atlete), non un cambiamento nella resa per partecipante.
- La quota femminile del medagliere varia molto tra paesi: alcuni costruiscono sulle donne più della metà delle proprie medaglie, e il confronto tra i periodi 1964-1996 e 2000-2020 mostra che quasi tutti i paesi si sono spostati verso l'alto, ma con velocità molto diverse.
- Le correlazioni con gli indicatori socio-economici sono **molto simili per i due generi**: dimensione economica (PIL) e demografica (popolazione) restano i fattori più associati al numero di medaglie per entrambi. Le divergenze tra generi sono piccole rispetto alla forza dei fattori comuni.
- Gli indicatori si associano debolmente anche alla *composizione* del medagliere (quota femminile): la ricchezza spiega quanto si vince, molto meno *con chi* si vince.

**Domande aperte** — quello che questi dati non riescono a spiegare:

1. **Perché alcuni paesi con condizioni socio-economiche simili hanno quote femminili molto diverse?** Gli indicatori World Bank sono generali: non misurano gli investimenti specifici nello sport femminile, le politiche federali, né fattori culturali e religiosi che influenzano l'accesso delle donne allo sport agonistico.
2. **Quanto pesa l'eredità dei sistemi sportivi statali?** URSS, DDR e poi Cina hanno investito presto e molto sullo sport femminile; la loro impronta sulla quota femminile è visibile ma non è spiegabile con gli indicatori economici a disposizione.
3. **Quanta parte della crescita è "meccanica"?** Il numero di eventi femminili in programma è deciso dal CIO: la quota femminile può crescere anche senza alcun cambiamento nei singoli paesi. Servirebbe normalizzare per il numero di eventi disponibili per genere in ogni edizione.
4. **I boicottaggi (1980, 1984) e le squadre unificate/miste** distorcono alcune edizioni: un'analisi robusta dovrebbe trattarle separatamente.
5. **Correlazione non è causalità**: paesi ricchi vincono di più, ma non sappiamo se la ricchezza *produca* successo femminile o se entrambi dipendano da altro (istituzioni, istruzione, urbanizzazione). Un modello multivariato con effetti fissi per paese sarebbe il passo successivo naturale.

**Limiti tecnici**: il genere dell'evento è ricavato dal nome dell'evento; gli eventi misti sono esclusi dalle quote; le medaglie di squadra sono contate una volta per squadra.

## 8. Esportazione dei grafici Altair per Jekyll

Questa cella esporta tutti i grafici del notebook in formato JSON Vega-Lite dentro `src/stefano/charts_definitivo`, richiamabili dal sito con il tag `vegachart`:

```liquid
{% vegachart /assets/charts/01_partecipazione_per_sesso.json %}
```

In [18]:
CHARTS_DIR = PROJECT_ROOT / "src" / "stefano" / "charts_definitivo"
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

charts_da_esportare = {
    "01_partecipazione_per_sesso.json": partecipazione_chart,
    "02_quota_atlete_vs_quota_medaglie.json": quote_confronto_chart,
    "03_composizione_medagliere_area.json": area_genere_chart,
    "04_resa_medaglie_per_100_atleti.json": resa_chart,
    "05_quota_femminile_per_paese.json": paesi_quota_chart,
    "06_quota_femminile_prima_dopo_2000.json": periodi_scatter,
    "07_correlazioni_per_genere_dot.json": corr_dot_chart,
    "08_differenza_correlazioni.json": corr_diff_chart,
    "09_correlazioni_quota_femminile.json": quota_corr_chart,
    "10_scatter_pil_pro_capite_regressione.json": pil_scatter_reg,
    "11_scatter_quota_femminile_pil.json": quota_pil_scatter,
}

for filename, chart in charts_da_esportare.items():
    chart.save(CHARTS_DIR / filename)

print(f"Esportati {len(charts_da_esportare)} grafici in: {CHARTS_DIR}")

Esportati 11 grafici in: C:\Users\smacchiavelli\development\code\master\progettone\src\stefano\charts_definitivo
